In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path



In [ ]:
DATA = Path("../data")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)



In [ ]:
# ======================
# 1. ЗАГРУЗКА ТЕКСТОВ
# ======================
df = pd.read_csv(DATA / "toxic_comments_sample.csv")



In [ ]:
print("rows:", df.shape[0])
print("toxic rate:", round(df["toxic"].mean(), 3))
df.head()



In [ ]:
df.info()



In [ ]:
# Баланс классов
counts = df["toxic"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
sns.barplot(
    x=["clean", "toxic"],
    y=counts.values,
    hue=["clean", "toxic"],
    palette="viridis",
    legend=False,
)
plt.title("Class balance", fontsize=14, pad=12)
plt.ylabel("Count")
for i, c in enumerate(counts.values):
    plt.text(i, c + 1, str(c), ha="center")
plt.tight_layout()
plt.show()



In [ ]:
def top_words(mask, n=15):
    # Простая токенизация по пробелам (без сложного NLP-пайплайна)
    tokens = " ".join(df.loc[mask, "text"].astype(str)).lower().split()
    return Counter(tokens).most_common(n)



In [ ]:
clean_top = top_words(df["toxic"] == 0)
toxic_top = top_words(df["toxic"] == 1)
print("clean:", clean_top[:8])
print("toxic:", toxic_top[:8])



In [ ]:
# ======================
# 2. СРАВНИТЕЛЬНЫЕ BARH
# ======================
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, label, items in zip(
    axes,
    ["clean", "toxic"],
    [clean_top, toxic_top],
):
    words = [w for w, _ in items][::-1]
    freqs = [c for _, c in items][::-1]
    # Горизонтальные столбцы: самые частые слова сверху после reverse
    ax.barh(words, freqs, color=sns.color_palette("viridis", len(words)))
    ax.set_title(label, fontsize=13)
    ax.set_xlabel("Frequency")

plt.suptitle("Top word frequencies by class", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIG / "nlp_top_words.png", dpi=120, bbox_inches="tight")
plt.show()



In [ ]:
# Облако частот (все тексты)
all_top = top_words(slice(None), n=20)
all_df = pd.DataFrame(all_top, columns=["word", "count"])

plt.figure(figsize=(9, 5))
sns.barplot(data=all_df, x="count", y="word", palette="viridis", orient="h")
plt.title("Top 20 words overall", fontsize=14, pad=12)
plt.tight_layout()
plt.show()



In [ ]:
# Длина документов
df = df.copy()
df["n_tokens"] = df["text"].astype(str).str.split().str.len()

plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1)
sns.histplot(data=df, x="n_tokens", hue="toxic", bins=15, palette="viridis")
plt.title("Token count distribution")
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="toxic", y="n_tokens", palette="viridis")
plt.title("Tokens by class")
plt.tight_layout()
plt.show()



In [ ]:
# Уникальные слова по классам
clean_vocab = set(" ".join(df.loc[df.toxic == 0, "text"].astype(str)).lower().split())
toxic_vocab = set(" ".join(df.loc[df.toxic == 1, "text"].astype(str)).lower().split())

only_toxic = sorted(toxic_vocab - clean_vocab)
only_clean = sorted(clean_vocab - toxic_vocab)
print("only toxic sample:", only_toxic[:20])
print("only clean sample:", only_clean[:20])



In [ ]:
# Размеры словарей
vocab_stats = pd.Series({
    "clean_vocab": len(clean_vocab),
    "toxic_vocab": len(toxic_vocab),
    "intersection": len(clean_vocab & toxic_vocab),
    "only_toxic": len(only_toxic),
    "only_clean": len(only_clean),
})
vocab_stats



In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x=vocab_stats.index, y=vocab_stats.values, palette="viridis")
plt.title("Vocabulary stats", fontsize=14, pad=12)
plt.xticks(rotation=20)
plt.ylabel("Count")
plt.tight_layout()
plt.show()



In [ ]:
# Таблица топ-слов toxic
pd.DataFrame(toxic_top, columns=["word", "count"])

